# Day 9 补充分析：每人次旅游收入、城乡居民收入对比
对应研究计划 Day 9

**这些是"顺带看一看"的补充内容，不是研究核心**——不能用这里算出的数字去说
"全市旅游收入证明了泰山旅游拉动了本地经济"这种话，全市数据和景区数据是不同范围。

运行方法：从上到下依次运行每个格子。

## 第一步：读取数据

In [1]:
import pandas as pd

df = pd.read_csv('../data/cleaned/analysis_data.csv')  # ← 需要改：换成实际路径
CORE_YEARS = [2019, 2020, 2021, 2022, 2023, 2024]

def get_series(name):
    s = df[df['指标名称'] == name].set_index('年份')['数值']
    return pd.to_numeric(s, errors='coerce').reindex(CORE_YEARS)

domestic_visitors = get_series('国内游客人次')          # 万人次
domestic_revenue  = get_series('国内旅游收入')          # 亿元
urban_income      = get_series('城镇居民人均可支配收入')  # 元
rural_income      = get_series('农村居民人均可支配收入')  # 元
income_ratio      = get_series('城乡居民收入比(农村=1)')  # 城镇/农村
cpi               = get_series('居民消费价格指数CPI涨幅')

print("数据读取完成")

数据读取完成


## 第二步：每人次国内旅游收入
公式：`收入(亿元) / 人次(万人次) * 10000 = 元/人次`
（亿=10⁸，万=10⁴，相除后还差10⁴才回到"元"，所以乘10000）

**注意**：这是全市口径，不是泰山景区一个点的情况，不能拿来说"每个来泰山的游客花了多少钱"。

In [2]:
per_capita_revenue = domestic_revenue / domestic_visitors * 10000
print("每人次国内旅游收入(元/人次)：", per_capita_revenue.round(1).to_dict())

每人次国内旅游收入(元/人次)： {2019: 1087.7, 2020: 956.5, 2021: 1014.8, 2022: 901.6, 2023: 944.9, 2024: 972.0}


## 第三步：城乡收入——倍数在缩小，但绝对差距呢？
城乡居民收入比这些年确实在下降，但这不代表城镇和农村的收入差距在"缩小"——
两边的收入本身都在涨，谁涨得更多，才是绝对差距变化的关键。

In [3]:
absolute_gap = urban_income - rural_income  # 元
print("城乡居民收入比(城镇/农村)：", income_ratio.to_dict())
print("城乡收入绝对差(城镇-农村，元)：", absolute_gap.to_dict())

gap_change_pct = (absolute_gap.loc[2024] / absolute_gap.loc[2019] - 1) * 100
ratio_change = income_ratio.loc[2024] - income_ratio.loc[2019]
print(f"\n2019→2024：收入比从{income_ratio.loc[2019]}降到{income_ratio.loc[2024]}"
      f"(变化{ratio_change:.2f})，但绝对差距从{absolute_gap.loc[2019]}元涨到"
      f"{absolute_gap.loc[2024]}元(涨了{gap_change_pct:.1f}%)")
print("——这说明「倍数缩小」和「绝对差距缩小」是两回事，不能只看比值就说城乡差距在改善。")

城乡居民收入比(城镇/农村)： {2019: 2.02, 2020: 1.98, 2021: 1.92, 2022: 1.88, 2023: 1.84, 2024: 1.82}
城乡收入绝对差(城镇-农村，元)： {2019: 19074, 2020: 19219, 2021: 19972, 2022: 20286, 2023: 20990, 2024: 21625}

2019→2024：收入比从2.02降到1.82(变化-0.20)，但绝对差距从19074元涨到21625元(涨了13.4%)
——这说明「倍数缩小」和「绝对差距缩小」是两回事，不能只看比值就说城乡差距在改善。


## 第四步：城镇/农村各自的名义增长率
看看是不是农村涨得比城镇快，这才是收入比缩小的真正原因。

In [4]:
urban_growth = urban_income.pct_change() * 100
rural_growth = rural_income.pct_change() * 100
print("城镇居民收入同比增长率(%,名义)：", urban_growth.round(1).to_dict())
print("农村居民收入同比增长率(%,名义)：", rural_growth.round(1).to_dict())

城镇居民收入同比增长率(%,名义)： {2019: nan, 2020: 3.2, 2021: 7.3, 2022: 4.1, 2023: 5.8, 2024: 4.7}
农村居民收入同比增长率(%,名义)： {2019: nan, 2020: 5.7, 2021: 10.6, 2022: 6.3, 2023: 7.8, 2024: 6.1}


## 第五步：这几年CPI水平——名义和实际差别大不大

In [5]:
print("CPI涨幅(%)：", (cpi*100).round(2).to_dict())
print("2023、2024年CPI涨幅接近0甚至为负，说明这两年名义收入增长和实际购买力增长差别很小，"
      "不需要像其他年份那样特别强调名义/实际的区别。")

CPI涨幅(%)： {2019: 2.4, 2020: 2.6, 2021: 0.9, 2022: 1.3, 2023: -0.1, 2024: -0.1}
2023、2024年CPI涨幅接近0甚至为负，说明这两年名义收入增长和实际购买力增长差别很小，不需要像其他年份那样特别强调名义/实际的区别。
